# Objectif du notebook 

Date de création : 20/01/2025

Notebook a run pour mettre en forme les données de la campagne Fiberscope Groix 2025 :

* Données AIS 
* Données GPS 
* Données SBE OBS 
* Fond de carte bathy pour les représentations 

In [1]:
import os
import sys
import numpy as np
import pandas as pd

In [2]:
project_root = os.path.abspath(
    os.path.join(os.path.dirname(os.getcwd()), "..", "..", "..")
)
root_groix_data = os.path.join(project_root, "data", "fiberscope_groix_oct_2025")
root_groix_metadata = os.path.join(root_groix_data, "metadata")
root_groix_ais = os.path.join(root_groix_data, "ais")
if not os.path.exists(root_groix_ais):
    os.makedirs(root_groix_ais)
root_groix_gps = os.path.join(root_groix_data, "gps")
if not os.path.exists(root_groix_gps):
    os.makedirs(root_groix_gps)
root_bathy_data = os.path.join(project_root, "data", "bathy")

ais_spationav_fpath = os.path.join(root_groix_ais, "SPATIONAV_AIS_001.csv")
root_gps_not_parsed = r"C:\Users\baptiste.menetrier\Desktop\ressource\XP_Fiberscope_Groix_092025\Jules\gps"

# Current folder
root_folder = os.path.join(project_root, "real_data_analysis", "fiberscope_groix")
data_folder = os.path.join(root_folder, "data")
img_folder = os.path.join(root_folder, "img")

In [3]:
sys.path.append(project_root)

from real_data_analysis.fiberscope_groix.src.data_prepocessing.preprocess_utils import (
    set_apriori_pos_h,
    load_apriori_pos_wgs84,
    plot_SBE39_obs,
    plot_SBE37_sounding,
    plot_bathy,
    build_SBE39_obs_dataset,
    build_oceano_src_dataset,
    build_SBE37_sounding_dataset,
    build_AIS_dataset,
    build_GPS_dataset,
    build_BATHY_dataset,
)

In [4]:
# sbe37_data_folder = os.path.join(root_groix_data, "SBE37_SOUNDING")
# interpolation_time_step = "30s"
# ds_SBE = build_SBE37_sounding_dataset(
#     sbe_data_folder=sbe37_data_folder,
#     interpolation_time_step=interpolation_time_step,
# )

# Données capteurs océano sur la source 

**RBR** : données dans un fichier txt, base de temps UTC avec potentiellement quelques secondes de décalage. 
**SBE39** : données du capteurs SBE39 sur la source. 



In [5]:
sbe39_src_data_folder = os.path.join(root_groix_data, "SBE39_SOURCE")
rbr_src_data_folder = os.path.join(root_groix_data, "RBR")

interpolation_time_step = "1s"
ds_SBE = build_oceano_src_dataset(
    sbe_data_folder=sbe39_src_data_folder,
    rbr_data_folder=rbr_src_data_folder,
    interpolation_time_step=interpolation_time_step,
)

*END*



# Données SBE des OBS 

In [ ]:
sbe39_obs_data_folder = os.path.join(root_groix_data, "SBE39_OBS")
interpolation_time_step = "30s"
ds_SBE = build_SBE39_obs_dataset(
    sbe_data_folder=sbe39_obs_data_folder,
    interpolation_time_step=interpolation_time_step,
)

In [ ]:
plot_SBE39_obs(ds_SBE=ds_SBE)

# Positions a priori des OBS

In [ ]:
N_geoid_undulation = 49.6795
apriori_pos_wgs84_before_campaign, apriori_pos_wgs84 = load_apriori_pos_wgs84(
    root_groix_metadata=root_groix_metadata
)
set_apriori_pos_h(
    apriori_pos_wgs84=apriori_pos_wgs84,
    N_geoid_undulation=N_geoid_undulation,
    ds_SBE=ds_SBE,
    verbose=False,
)
set_apriori_pos_h(
    apriori_pos_wgs84=apriori_pos_wgs84_before_campaign,
    N_geoid_undulation=N_geoid_undulation,
    ds_SBE=ds_SBE,
    verbose=False,
)

In [ ]:
apriori_pos_wgs84

In [ ]:
# Coordonnées de le l'origine du repère de référence
obs_pos_id = "obs2"
pos0 = apriori_pos_wgs84.loc[obs_pos_id]
# lat0 = np.radians(pos0.lat)
# lon0 = np.radians(pos0.lon)
lat0 = pos0.lat
lon0 = pos0.lon
h0 = pos0.h

local_frame_origin = {
    "id": obs_pos_id,
    "lat": lat0,
    "lon": lon0,
    "h": h0,
}

# Données bathy pour le fond de carte 

La zone sélectionnée est une zone de 0.25° x 0.25° soit environ 37.56 km x 55.6 km 

In [ ]:
ds_bathy = build_BATHY_dataset(local_frame_origin, root_bathy_data=root_bathy_data)
plot_bathy(ds_bathy, contour_levels=[0])

# Données GPS / AIS  

In [ ]:
time_step = "10s"  # 10 seconds
t_start = pd.Timestamp("2025-10-13 08:00:00", tz="UTC")     # Min AIS 
t_end = pd.Timestamp("2025-10-17 09:47:50", tz="UTC")       # Max AIS

In [ ]:
ds_ais = build_AIS_dataset(
    local_frame_origin=local_frame_origin,
    N_geoid_undulation=N_geoid_undulation,
    t_start=t_start,
    t_end=t_end,
    ds_SBE=ds_SBE,
    interpolation_time_step=time_step,
    ais_spationav_fpath=ais_spationav_fpath,
    root_groix_metadata=root_groix_metadata,
)

ds_gps = build_GPS_dataset(
    local_frame_origin=local_frame_origin,
    N_geoid_undulation=N_geoid_undulation,
    t_start=t_start,
    t_end=t_end,
    ds_SBE=ds_SBE,
    interpolation_time_step=time_step,
    root_groix_gps=root_groix_gps,
    root_groix_metadata=root_groix_metadata,
)

# Données sondage SBE 

Données des sondages SBE réalisés dans la semaine. 

* La SBE24 est inutilisable (données erratiques).
* Deux SBE37 ont été utilisés : SBE37_20453 et SBE37_23773
 
Les données lisibles sont stockées dans les fichiers .cnv, les grandeurs associées à chaque colonne sont décrites dans l’en-tête du fichier (name 0 à 9).


In [ ]:
sbe37_data_folder = os.path.join(root_groix_data, "SBE37_SOUNDING")
interpolation_time_step = "30s"
ds_SBE = build_SBE37_sounding_dataset(
    sbe_data_folder=sbe37_data_folder,
    interpolation_time_step=interpolation_time_step,
)

In [ ]:
plot_SBE39_obs(ds_SBE=ds_SBE)

# Sauvergarde de l'ensemble des données au format NETCDF

In [ ]:
# Save datasets to netcdf files
ds_gps.to_netcdf(os.path.join(data_folder, "gps.nc"))
ds_ais.to_netcdf(os.path.join(data_folder, "ais.nc"))
ds_bathy.to_netcdf(os.path.join(data_folder, "bathy.nc"))
ds_SBE.to_netcdf(os.path.join(data_folder, "sbe39_obs.nc"))